# AI-Task9: Modular News Briefing Builder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaweriamushtaq41-coder/Smart_District_2/blob/main/AI-Task9/AI_Task9_News_Briefing_Builder.ipynb)

**Kya banana hai:** Sirf official Runnable classes: `RunnableParallel` (headline + sentiment ek saath), `RunnableLambda` (jodna aur format), `RunnableGenerator` (briefing jumla ba jumla stream), sab `RunnableSequence` mein `|` se. `RunnableConfig` se tag aur callback (har step ka log).

**Concepts:** RunnableParallel, RunnableLambda, RunnableGenerator, RunnableSequence, RunnableConfig, callbacks

**Colab mein kaise chalayen:**
1. Upar *Open In Colab* dabao (ya ye file Colab mein upload karo).
2. *Runtime > Run all* dabao.
3. Keys ho to Colab ke left side ke **Secrets** (chaabi ka icon) mein daalo. Bina key ke bhi chalta hai (**demo mode**, nakli jawab).

> Notebook ko share karne se pehle apni keys kabhi output / cell mein mat chhodo.

## 1. Libraries install aur keys (`.env`)

In [1]:
!pip -q install -U python-dotenv langchain-core langchain-openai langchain-anthropic langchain-google-genai langchain-groq

In [2]:
import os

# .env file banao. Keys do tareeqon se aa sakti hain:
#  1) Colab "Secrets" (left side ka chaabi wala icon), secret ka naam wahi jo neeche list mein hai, "Notebook access" ON
#  2) apni .env file Colab ke "Files" panel (left side ka folder icon) mein drag-drop kar do, run karne se pehle
NAMES = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"]

existing = {}                       # pehle se .env ho (upload ki hui) to uski keys bacha lo
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                key, value = line.strip().split("=", 1)
                existing[key] = value

lines = []
for name in NAMES:
    value = existing.get(name, "")
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:          # Colab nahi hai, ya ye secret nahi banaya
        pass
    lines.append(f"{name}={value}")

with open(".env", "w") as f:
    f.write("\n".join(lines) + "\n")

from dotenv import load_dotenv
load_dotenv(".env", override=True)

found = [n for n in NAMES if os.getenv(n)]
print("Keys mili:", found if found else "koi nahi (demo mode chalega)")

Keys mili: koi nahi (demo mode chalega)


## 2. Model factory
`get_chat_model("groq")` jaisa call asli LangChain model deta hai (ChatOpenAI, ChatAnthropic, ChatGoogleGenerativeAI, ChatGroq).
Provider badalna ho to sirf naam badlo, chain wahi rehti hai. `demo_model` / `demo_stream_model` key ke baghair nakli model dete hain jo chain mein asli ki tarah lagte hain.

In [3]:
import os

from dotenv import load_dotenv
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

load_dotenv()

PROVIDERS = ["openai", "claude", "gemini", "groq"]

KEY_NAMES = {
    "openai": "OPENAI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
}

MODELS = {
    "openai": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    "claude": os.getenv("ANTHROPIC_MODEL", "claude-3-5-haiku-latest"),
    "gemini": os.getenv("GEMINI_MODEL", "gemini-1.5-flash"),
    "groq": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
}


def available_providers():
    """Jin providers ki key .env mein hai."""
    return [p for p in PROVIDERS if os.getenv(KEY_NAMES[p], "").strip()]


def get_chat_model(provider, temperature=0.3):
    """Asli LangChain model wapas deta hai. Import andar hai taake sirf wahi package chahiye jo use ho."""
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODELS["openai"], temperature=temperature)
    if provider == "claude":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=MODELS["claude"], temperature=temperature)
    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODELS["gemini"], temperature=temperature)
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODELS["groq"], temperature=temperature)
    raise ValueError(f"Unknown provider: {provider}")


# ---------------- demo (nakli) models ----------------
def demo_stream_model(text, delay=0.004):
    """Ek fixed jawab deta hai, harf harf stream hota hai (asli streaming jaisa)."""
    return FakeListChatModel(responses=[text], sleep=delay)


def demo_model(responder):
    """responder(prompt_text) -> jawab ka text. Prompt dekh kar jawab chunta hai,
    is liye RunnableParallel mein (jahan tarteeb pakki nahi) bhi theek chalta hai."""
    def run(prompt_value):
        text = prompt_value if isinstance(prompt_value, str) else prompt_value.to_string()
        return AIMessage(content=responder(text))
    return RunnableLambda(run)

## 3. Chain ke hisse
- `StepLogger`: custom callback, har step `logs/news_briefing.log` mein likhta hai
- `merge_and_format` (RunnableLambda): headline + sentiment ko briefing text banata hai
- `stream_sentences` (RunnableGenerator): text ke tukray leta hai aur poora jumla banne par yield karta hai
- `build_chain`: `to_dict | parallel | merge | generator` (khud `RunnableSequence` ban jata hai)

In [4]:
import logging
import os
import re
from typing import Iterator

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (RunnableConfig, RunnableGenerator, RunnableLambda,
                                      RunnableParallel)

In [5]:
log = logging.getLogger("news_briefing")


def setup_logging():
    os.makedirs("logs", exist_ok=True)
    log.setLevel(logging.INFO)
    log.handlers.clear()
    handler = logging.FileHandler("logs/news_briefing.log", encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))
    log.addHandler(handler)


class StepLogger(BaseCallbackHandler):
    """Custom callback: chain ka har step shuru / khatam hone par log likhta hai."""

    def on_chain_start(self, serialized, inputs, *, run_id, tags=None, **kwargs):
        log.info("START %-22s tags=%s", kwargs.get("name") or "chain", tags)

    def on_chain_end(self, outputs, *, run_id, tags=None, **kwargs):
        log.info("END   %s", str(outputs)[:70].replace("\n", " "))

    def on_chain_error(self, error, *, run_id, **kwargs):
        log.error("ERROR %s", error)


headline_prompt = ChatPromptTemplate.from_template(
    "Write ONE sentence summarising the main news headline about: {topic}")
sentiment_prompt = ChatPromptTemplate.from_template(
    "Give the overall news sentiment about '{topic}' as a single number from -1.0 (very negative) "
    "to 1.0 (very positive). Reply with the number only.")


def merge_and_format(data):
    """RunnableLambda: headline + sentiment ko briefing text mein badalta hai."""
    found = re.search(r"-?\d+(?:\.\d+)?", data["sentiment"])
    score = max(-1.0, min(1.0, float(found.group()))) if found else 0.0     # number na mile to neutral
    label = "positive" if score > 0.2 else "negative" if score < -0.2 else "neutral"
    return (f"Today's briefing on {data['topic']}. "
            f"Headline: {data['headline'].strip()} "
            f"The overall sentiment is {label} ({score:+.2f}). "
            f"That is all for this briefing.")


def stream_sentences(chunks: Iterator[str]) -> Iterator[str]:
    """RunnableGenerator ka function: text ke tukray leta hai, poora jumla banne par yield karta hai."""
    buffer = ""
    for chunk in chunks:
        buffer += chunk
        while True:
            match = re.search(r"(?<=[.!?])\s+", buffer)         # jumla khatam hone ki jagah
            if not match:
                break
            yield buffer[:match.start()].strip()
            buffer = buffer[match.end():]
    if buffer.strip():
        yield buffer.strip()


def build_chain(llm):
    parallel = RunnableParallel(
        headline=headline_prompt | llm | StrOutputParser(),
        sentiment=sentiment_prompt | llm | StrOutputParser(),
        topic=RunnableLambda(lambda d: d["topic"]),
    )
    to_dict = RunnableLambda(lambda topic: {"topic": topic.strip()})
    # | se jodne par ye khud RunnableSequence ban jata hai
    return to_dict | parallel | RunnableLambda(merge_and_format) | RunnableGenerator(stream_sentences)


# ---------------- demo (nakli model) ----------------
def demo_brain(prompt_text):
    topic = re.search(r"(?:about|on): (.+)|about '(.+?)'", prompt_text)
    topic = (topic.group(1) or topic.group(2)).strip() if topic else "the topic"
    if "single number" in prompt_text:
        return "0.45"
    return f"New developments in {topic} are drawing wide attention from the public and experts alike."

## 4. Provider chuno

In [6]:
FORCE_DEMO = False      # True likhoge to hamesha nakli model
PROVIDER = None         # "groq", "openai", "claude" ya "gemini". None = jis ki key mili
demo = FORCE_DEMO
provider = PROVIDER or (available_providers() or [None])[0]
if provider is None:
    print("[!] Koi API key nahi mili, DEMO mode chal raha hai.")
    demo, provider = True, "groq"
print("provider =", provider, "| demo =", demo)

[!] Koi API key nahi mili, DEMO mode chal raha hai.
provider = groq | demo = True


## 5. Chalao
`config` mein custom tag aur callback lagta hai. `chain.stream()` har jumla bante hi deta hai.

In [7]:
TOPIC = "AI in education"

llm = demo_model(demo_brain) if demo else get_chat_model(provider, temperature=0.3)
setup_logging()
chain = build_chain(llm)
print("Chain ki qisam:", type(chain).__name__, "\n")

config: RunnableConfig = {
    "tags": ["news-briefing"],                    # custom tag
    "callbacks": [StepLogger()],                  # custom callback (logging)
    "metadata": {"topic": TOPIC},
    "run_name": "news_briefing",
}

print("BRIEFING:")
for number, sentence in enumerate(chain.stream(TOPIC, config=config), start=1):
    print(f"  {number}. {sentence}", flush=True)

Chain ki qisam: RunnableSequence 

BRIEFING:
  1. Today's briefing on AI in education.
  2. Headline: New developments in AI in education are drawing wide attention from the public and experts alike.
  3. The overall sentiment is positive (+0.45).
  4. That is all for this briefing.


## 6. Log file dekho (har step)

In [8]:
print(open("logs/news_briefing.log", encoding="utf-8").read()[:1500])

2026-10-31 18:07:47,230 | START news_briefing          tags=['news-briefing']
2026-10-31 18:07:47,232 | START RunnableLambda         tags=['seq:step:1', 'news-briefing']
2026-10-31 18:07:47,233 | START RunnableParallel<headline,sentiment,topic> tags=['seq:step:2', 'news-briefing']
2026-10-31 18:07:47,234 | START RunnableSequence       tags=['map:key:headline', 'news-briefing']
2026-10-31 18:07:47,235 | END   {'topic': 'AI in education'}
2026-10-31 18:07:47,238 | START ChatPromptTemplate     tags=['seq:step:1', 'news-briefing']
2026-10-31 18:07:47,238 | END   messages=[HumanMessage(content='Write ONE sentence summarising the mai
2026-10-31 18:07:47,239 | START run                    tags=['seq:step:2', 'news-briefing']
2026-10-31 18:07:47,239 | START StrOutputParser        tags=['seq:step:3', 'news-briefing']
2026-10-31 18:07:47,243 | START RunnableSequence       tags=['map:key:sentiment', 'news-briefing']
2026-10-31 18:07:47,244 | START ChatPromptTemplate     tags=['seq:step:1', 'news-

## Kya seekha
- Chain `RunnableSequence` hai, `|` se jodne par khud ban jati hai.
- `RunnableGenerator` streaming ke liye hai: function chunks leta hai aur cheezein yield karta hai.
- `RunnableConfig` mein `tags`, `callbacks`, `metadata` dete hain; callback se har step ka log milta hai.